In [1]:
import torch
import torch.nn as nn
import timm
import pandas as pd
import numpy as np
import os
import math
import json
from PIL import Image
from collections import Counter
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from sklearn.model_selection import StratifiedKFold
from tqdm import tqdm

<jemalloc>: Unsupported system page size


In [2]:
GPU_IDS     = [1,2]                          # ← hier deine zwei GPUs eintragen
device      = torch.device(f"cuda:{GPU_IDS[0]}" if torch.cuda.is_available() else "cpu")
USE_MULTI_GPU = torch.cuda.is_available() and len(GPU_IDS) > 1

IMG_SIZE    = 384
BATCH_SIZE  = 32          # ← verdoppelt (16 pro GPU × 2 GPUs)
ACCUM_STEPS = 2           # ← halbiert, da effektive Batch-Size gleich bleibt
BACKBONE    = "convnextv2_huge.fcmae_ft_in22k_in1k_384"
NUM_WORKERS = 8           # ← erhöht für 2 GPUs
SEED        = 42
PATIENCE    = 3

MEAN = [0.485, 0.456, 0.406]
STD  = [0.229, 0.224, 0.225]

torch.manual_seed(SEED)
np.random.seed(SEED)
print(f"Device: {device} | Backbone: {BACKBONE}")
print(f"GPUs genutzt: {GPU_IDS} | Multi-GPU: {USE_MULTI_GPU}")
for gid in GPU_IDS:
    name = torch.cuda.get_device_name(gid)
    mem  = torch.cuda.get_device_properties(gid).total_memory / 1e9
    print(f"  GPU {gid}: {name} ({mem:.1f} GB)")


Device: cuda:1 | Backbone: convnextv2_huge.fcmae_ft_in22k_in1k_384
GPUs genutzt: [1, 2] | Multi-GPU: True
  GPU 1: Tesla V100-SXM2-32GB (33.8 GB)
  GPU 2: Tesla V100-SXM2-32GB (33.8 GB)


In [3]:
data_dir   = "/datasets/multi-view-pig-posture-recognition/"
train2_df  = pd.read_csv(os.path.join(data_dir, "train2.csv"))
dir_train2 = os.path.join(data_dir, "train2_images")
test_df    = pd.read_csv(os.path.join(data_dir, "test.csv"))
dir_test   = os.path.join(data_dir, "test_images")

In [4]:
def parse_bbox(x):
    if isinstance(x, str):
        try:    return json.loads(x)
        except: return [float(i) for i in x.replace("[","").replace("]","").split(",")]
    return x

train2_df["bbox"] = train2_df["bbox"].apply(parse_bbox)
test_df["bbox"]   = test_df["bbox"].apply(parse_bbox)

def extract_cam(image_id):
    basename = os.path.basename(str(image_id))
    for part in basename.replace("-","_").split("_"):
        if part.lower().startswith("cam"):
            return part.lower()
    return "cam_unknown"

train2_df["camera"] = train2_df["image_id"].apply(extract_cam)
test_df["camera"]   = test_df["image_id"].apply(extract_cam)

print("Kamera-Verteilung Train:")
print(train2_df["camera"].value_counts().sort_index())
print("\nKamera-Verteilung Test:")
print(test_df["camera"].value_counts().sort_index())

all_cams   = sorted(train2_df["camera"].unique())
half       = len(all_cams) // 2
cam_groupA = all_cams[:half]
cam_groupB = all_cams[half:]
print(f"\nModell A → Kameras: {cam_groupA}")
print(f"Modell B → Kameras: {cam_groupB}")

train_A = train2_df[train2_df["camera"].isin(cam_groupA)].copy()
train_B = train2_df[train2_df["camera"].isin(cam_groupB)].copy()
print(f"\nTrain A: {len(train_A)} | Train B: {len(train_B)}")

train_labels = train2_df["class_id"].tolist()
class_counts = Counter(train_labels)
num_classes  = len(class_counts)
total        = len(train_labels)
print(f"Klassen: {num_classes}")


Kamera-Verteilung Train:
camera
cam1    13452
cam2     9998
Name: count, dtype: int64

Kamera-Verteilung Test:
camera
cam1    4264
cam2    7444
Name: count, dtype: int64

Modell A → Kameras: ['cam1']
Modell B → Kameras: ['cam2']

Train A: 13452 | Train B: 9998
Klassen: 5


In [5]:
def make_split(df):
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
    tr_idx, va_idx = next(skf.split(df, df["class_id"]))
    return df.iloc[tr_idx], df.iloc[va_idx]

train_A_tr, train_A_va = make_split(train_A)
train_B_tr, train_B_va = make_split(train_B)
print(f"A → Train: {len(train_A_tr)} | Val: {len(train_A_va)}")
print(f"B → Train: {len(train_B_tr)} | Val: {len(train_B_va)}")

def crop_pig(image, bbox, expand=1.15):
    x, y, w, h   = bbox
    cx, cy       = x + w/2, y + h/2
    img_w, img_h = image.size
    x_min = max(0, int(cx - w*expand/2))
    y_min = max(0, int(cy - h*expand/2))
    x_max = min(img_w, int(cx + w*expand/2))
    y_max = min(img_h, int(cy + h*expand/2))
    if x_max > x_min and y_max > y_min:
        image = image.crop((x_min, y_min, x_max, y_max))
    return image

A → Train: 10761 | Val: 2691
B → Train: 7998 | Val: 2000


In [6]:
class PigDataset(Dataset):
    def __init__(self, df, img_dir, transform=None, is_test=False):
        self.df        = df.reset_index(drop=True)
        self.img_dir   = img_dir
        self.transform = transform
        self.is_test   = is_test

    def __len__(self): return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(os.path.join(self.img_dir, row["image_id"])).convert("RGB")
        img = crop_pig(img, row["bbox"])
        if self.transform: img = self.transform(img)
        if self.is_test:   return img, str(row["row_id"])
        return img, int(row["class_id"])

In [7]:
train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE + 32, IMG_SIZE + 32)),
    transforms.RandomCrop(IMG_SIZE),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(p=0.1),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.15, hue=0.03),
    transforms.RandomRotation(degrees=15),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
    transforms.RandomErasing(p=0.2, scale=(0.02, 0.12)),
])

val_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

In [8]:
def make_loaders(train_df, val_df):
    tr = DataLoader(
        PigDataset(train_df, dir_train2, train_transform),
        batch_size=BATCH_SIZE, shuffle=True,
        num_workers=NUM_WORKERS, pin_memory=True, drop_last=True
    )
    va = DataLoader(
        PigDataset(val_df, dir_train2, val_transform),
        batch_size=BATCH_SIZE, shuffle=False,
        num_workers=NUM_WORKERS, pin_memory=True
    )
    return tr, va

# ── Loss ──
class_weights_tensor = torch.FloatTensor(
    [total / (num_classes * class_counts[i]) for i in range(num_classes)]
).to(device)
criterion = nn.CrossEntropyLoss(weight=class_weights_tensor, label_smoothing=0.1)


In [9]:
class PigModel(nn.Module):
    def __init__(self, num_classes, backbone_name=BACKBONE):
        super().__init__()
        self.backbone = timm.create_model(backbone_name, pretrained=True, num_classes=0)
        self.backbone.requires_grad_(False)
        feat_dim = self.backbone.num_features
        print(f"  Backbone: {backbone_name} | Feature dim: {feat_dim}")

        self.head = nn.Sequential(
            nn.LayerNorm(feat_dim),
            nn.Dropout(0.4),
            nn.Linear(feat_dim, 512),
            nn.GELU(),
            nn.Dropout(0.2),
            nn.Linear(512, num_classes),
        )

    def forward(self, x):
        return self.head(self.backbone(x))

    def unfreeze(self, fraction=0.0):
        # Bei DataParallel über .module auf das eigentliche Modell zugreifen
        backbone = self.backbone
        params   = list(backbone.parameters())
        for p in params: p.requires_grad = False
        if fraction > 0:
            n = int(len(params) * fraction)
            for p in params[-n:]: p.requires_grad = True
            print(f"  Unfroze {n}/{len(params)} Backbone-Params ({fraction*100:.0f}%)")


In [10]:
def wrap_model(model):
    """Wrapped Modell mit DataParallel falls mehrere GPUs verfügbar."""
    model = model.to(device)
    if USE_MULTI_GPU:
        model = nn.DataParallel(model, device_ids=GPU_IDS)
        print(f"  ✅ DataParallel aktiviert auf GPUs: {GPU_IDS}")
    return model


def unwrap_model(model):
    """Gibt das eigentliche Modell zurück (ohne DataParallel-Wrapper)."""
    return model.module if isinstance(model, nn.DataParallel) else model


In [11]:
class CosineWarmupScheduler:
    def __init__(self, optimizer, warmup_epochs, max_epochs, min_lr_ratio=0.01):
        self.optimizer     = optimizer
        self.warmup_epochs = warmup_epochs
        self.max_epochs    = max_epochs
        self.min_lr_ratio  = min_lr_ratio
        self.current_epoch = 0
        for pg in optimizer.param_groups:
            pg['initial_lr'] = pg['lr']

    def step(self):
        self.current_epoch += 1
        e = self.current_epoch
        if e <= self.warmup_epochs:
            scale = e / self.warmup_epochs
        else:
            progress = (e - self.warmup_epochs) / (self.max_epochs - self.warmup_epochs)
            scale = self.min_lr_ratio + 0.5*(1-self.min_lr_ratio)*(1+math.cos(math.pi*progress))
        for pg in self.optimizer.param_groups:
            pg['lr'] = pg['initial_lr'] * scale

    def get_lr(self): return self.optimizer.param_groups[0]['lr']


class EarlyStopping:
    def __init__(self, patience=3, min_delta=1e-4):
        self.patience    = patience
        self.min_delta   = min_delta
        self.best_acc    = 0.0
        self.counter     = 0
        self.should_stop = False

    def step(self, val_acc):
        if val_acc > self.best_acc + self.min_delta:
            self.best_acc = val_acc
            self.counter  = 0
        else:
            self.counter += 1
            print(f"  ⚠️  Keine Verbesserung ({self.counter}/{self.patience})")
            if self.counter >= self.patience:
                self.should_stop = True
                print(f"  🛑 Early Stopping! Beste Acc: {self.best_acc:.4f}")


In [12]:
def mixup(x, y, alpha=0.4):
    lam = np.random.beta(alpha, alpha)
    idx = torch.randperm(x.size(0), device=x.device)
    return lam*x + (1-lam)*x[idx], y, y[idx], lam

def mixup_loss(crit, pred, ya, yb, lam):
    return lam*crit(pred, ya) + (1-lam)*crit(pred, yb)

scaler = torch.cuda.amp.GradScaler()
def autocast(): return torch.cuda.amp.autocast()

def train_epoch(model, loader, optimizer, use_mixup=True):
    model.train()
    total_loss = 0.0
    optimizer.zero_grad()
    for i, (imgs, lbls) in enumerate(tqdm(loader, desc="  Train", leave=False)):
        imgs = imgs.to(device, non_blocking=True)
        lbls = lbls.to(device, non_blocking=True)

        if use_mixup and np.random.rand() < 0.5:
            imgs, ya, yb, lam = mixup(imgs, lbls)
            with autocast():
                loss = mixup_loss(criterion, model(imgs), ya, yb, lam) / ACCUM_STEPS
        else:
            with autocast():
                loss = criterion(model(imgs), lbls) / ACCUM_STEPS

        scaler.scale(loss).backward()
        if (i+1) % ACCUM_STEPS == 0:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()

        total_loss += loss.item() * ACCUM_STEPS
    return total_loss / len(loader)


In [14]:
@torch.no_grad()
def val_epoch(model, loader):
    from sklearn.metrics import f1_score
    model.eval()
    correct, total, val_loss = 0, 0, 0.0
    all_preds, all_labels = [], []
    for imgs, lbls in tqdm(loader, desc="  Val  ", leave=False):
        imgs = imgs.to(device, non_blocking=True)
        lbls = lbls.to(device, non_blocking=True)
        with autocast():
            out      = model(imgs)
            val_loss += criterion(out, lbls).item()
        preds = out.argmax(1)
        correct += (preds == lbls).sum().item()
        total   += lbls.size(0)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(lbls.cpu().numpy())
    macro_f1 = f1_score(all_labels, all_preds, average="macro")
    return val_loss / len(loader), correct / total, macro_f1


def run_phase(model, train_loader, val_loader, optimizer, scheduler,
              max_epochs, save_name, use_mixup=True, patience=PATIENCE):
    best_acc = 0.0
    es       = EarlyStopping(patience=patience)
    for epoch in range(max_epochs):
        tr_loss              = train_epoch(model, train_loader, optimizer, use_mixup)
        va_loss, va_acc, va_f1 = val_epoch(model, val_loader)
        lr              = scheduler.get_lr()
        scheduler.step()

        saved = ""
        if va_f1 > best_acc:
            best_acc = va_f1
            # unwrap_model: nur state_dict des eigentlichen Modells speichern
            torch.save(unwrap_model(model).state_dict(), save_name)
            saved = " ✅ gespeichert"

        print(f"  Ep {epoch+1:02d}/{max_epochs} | "
              f"TrL: {tr_loss:.4f} | VaL: {va_loss:.4f} | "
              f"Acc: {va_acc:.4f} | F1: {va_f1:.4f} | LR: {lr:.2e}{saved}")

        es.step(va_f1)
        if es.should_stop:
            break

    print(f"  → Bester Macro F1 dieser Phase: {best_acc:.4f}")
    return best_acc


def train_model(tag, train_tr, train_va):
    print(f"\n{'█'*55}")
    print(f"  MODELL {tag} | Kamera-Gruppe: {cam_groupA if tag=='A' else cam_groupB}")
    print(f"{'█'*55}")

    model = PigModel(num_classes=num_classes)
    model = wrap_model(model)   # ← DataParallel wrapping

    loader_tr, loader_va = make_loaders(train_tr, train_va)

    # ── Phase 1 ──
    print(f"\n── Phase 1: Head only ──")
    opt1 = torch.optim.AdamW(unwrap_model(model).head.parameters(), lr=1e-3, weight_decay=1e-2)
    sch1 = CosineWarmupScheduler(opt1, warmup_epochs=2, max_epochs=10)
    b1 = run_phase(model, loader_tr, loader_va, opt1, sch1,
                   max_epochs=1, save_name=f"cnv2_{tag}_p1.pth",
                   use_mixup=False, patience=3)

    # ── Phase 2 ──
    print(f"\n── Phase 2: 20% Backbone ──")
    unwrap_model(model).load_state_dict(torch.load(f"cnv2_{tag}_p1.pth", map_location=device))
    unwrap_model(model).unfreeze(0.20)
    opt2 = torch.optim.AdamW([
        {"params": [p for p in unwrap_model(model).backbone.parameters() if p.requires_grad], "lr": 5e-6},
        {"params": unwrap_model(model).head.parameters(), "lr": 5e-5},
    ], weight_decay=1e-2)
    sch2 = CosineWarmupScheduler(opt2, warmup_epochs=1, max_epochs=10)
    b2 = run_phase(model, loader_tr, loader_va, opt2, sch2,
                   max_epochs=1, save_name=f"cnv2_{tag}_p2.pth",
                   use_mixup=True, patience=3)

    # ── Phase 3 ──
    print(f"\n── Phase 3: 50% Backbone ──")
    unwrap_model(model).load_state_dict(torch.load(f"cnv2_{tag}_p2.pth", map_location=device))
    unwrap_model(model).unfreeze(0.50)
    opt3 = torch.optim.AdamW([
        {"params": [p for p in unwrap_model(model).backbone.parameters() if p.requires_grad], "lr": 1e-6},
        {"params": unwrap_model(model).head.parameters(), "lr": 1e-5},
    ], weight_decay=5e-3)
    sch3 = CosineWarmupScheduler(opt3, warmup_epochs=1, max_epochs=10)
    b3 = run_phase(model, loader_tr, loader_va, opt3, sch3,
                   max_epochs=1, save_name=f"cnv2_{tag}_p3.pth",
                   use_mixup=True, patience=3)

    best_file = max([(b1, f"cnv2_{tag}_p1.pth"),
                     (b2, f"cnv2_{tag}_p2.pth"),
                     (b3, f"cnv2_{tag}_p3.pth")])[1]

    print(f"\n  📊 Modell {tag}: P1={b1:.4f} | P2={b2:.4f} | P3={b3:.4f}")
    print(f"  🏆 Bestes Checkpoint: {best_file}")
    return best_file

best_A = train_model("A", train_A_tr, train_A_va)
best_B = train_model("B", train_B_tr, train_B_va)



███████████████████████████████████████████████████████
  MODELL A | Kamera-Gruppe: ['cam1']
███████████████████████████████████████████████████████
  Backbone: convnextv2_huge.fcmae_ft_in22k_in1k_384 | Feature dim: 2816
  ✅ DataParallel aktiviert auf GPUs: [1, 2]

── Phase 1: Head only ──


  Ep 01/1 | TrL: 1.4001 | VaL: 1.2622 | Acc: 0.7083 | F1: 0.5961 | LR: 1.00e-03 ✅ gespeichert
  → Bester Macro F1 dieser Phase: 0.5961

── Phase 2: 20% Backbone ──
  Unfroze 75/378 Backbone-Params (20%)


  Ep 01/1 | TrL: 1.2556 | VaL: 1.1329 | Acc: 0.7752 | F1: 0.6891 | LR: 5.00e-06 ✅ gespeichert
  → Bester Macro F1 dieser Phase: 0.6891

── Phase 3: 50% Backbone ──
  Unfroze 189/378 Backbone-Params (50%)


  Ep 01/1 | TrL: 1.1770 | VaL: 1.0725 | Acc: 0.7993 | F1: 0.7151 | LR: 1.00e-06 ✅ gespeichert
  → Bester Macro F1 dieser Phase: 0.7151

  📊 Modell A: P1=0.5961 | P2=0.6891 | P3=0.7151
  🏆 Bestes Checkpoint: cnv2_A_p3.pth

███████████████████████████████████████████████████████
  MODELL B | Kamera-Gruppe: ['cam2']
███████████████████████████████████████████████████████
  Backbone: convnextv2_huge.fcmae_ft_in22k_in1k_384 | Feature dim: 2816
  ✅ DataParallel aktiviert auf GPUs: [1, 2]

── Phase 1: Head only ──


  Ep 01/1 | TrL: 1.5607 | VaL: 1.4295 | Acc: 0.6630 | F1: 0.5224 | LR: 1.00e-03 ✅ gespeichert
  → Bester Macro F1 dieser Phase: 0.5224

── Phase 2: 20% Backbone ──
  Unfroze 75/378 Backbone-Params (20%)


  Ep 01/1 | TrL: 1.3922 | VaL: 1.2463 | Acc: 0.7520 | F1: 0.6487 | LR: 5.00e-06 ✅ gespeichert
  → Bester Macro F1 dieser Phase: 0.6487

── Phase 3: 50% Backbone ──
  Unfroze 189/378 Backbone-Params (50%)


  Ep 01/1 | TrL: 1.3225 | VaL: 1.2016 | Acc: 0.7235 | F1: 0.6347 | LR: 1.00e-06 ✅ gespeichert
  → Bester Macro F1 dieser Phase: 0.6347

  📊 Modell B: P1=0.5224 | P2=0.6487 | P3=0.6347
  🏆 Bestes Checkpoint: cnv2_B_p2.pth


In [14]:
print("\n" + "="*55)
print("TTA Ensemble Inferenz (Modell A + Modell B)")
print("="*55)

tta_transforms = [
    transforms.Compose([
        transforms.Resize((IMG_SIZE, IMG_SIZE)),
        transforms.ToTensor(), transforms.Normalize(MEAN, STD)]),
    transforms.Compose([
        transforms.Resize((IMG_SIZE, IMG_SIZE)),
        transforms.Lambda(lambda x: x.transpose(Image.FLIP_LEFT_RIGHT)),
        transforms.ToTensor(), transforms.Normalize(MEAN, STD)]),
    transforms.Compose([
        transforms.Resize((IMG_SIZE + 48, IMG_SIZE + 48)),
        transforms.CenterCrop(IMG_SIZE),
        transforms.ToTensor(), transforms.Normalize(MEAN, STD)]),
    transforms.Compose([
        transforms.Resize((IMG_SIZE, IMG_SIZE)),
        transforms.Lambda(lambda x: x.transpose(Image.FLIP_TOP_BOTTOM)),
        transforms.ToTensor(), transforms.Normalize(MEAN, STD)]),
    transforms.Compose([
        transforms.Resize((IMG_SIZE + 48, IMG_SIZE + 48)),
        transforms.CenterCrop(IMG_SIZE),
        transforms.Lambda(lambda x: x.transpose(Image.FLIP_LEFT_RIGHT)),
        transforms.ToTensor(), transforms.Normalize(MEAN, STD)]),
]



TTA Ensemble Inferenz (Modell A + Modell B)


In [15]:
class TestDataset(Dataset):
    def __init__(self, df, img_dir, transform):
        self.df = df.reset_index(drop=True)
        self.img_dir = img_dir
        self.transform = transform
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(os.path.join(self.img_dir, row["image_id"])).convert("RGB")
        img = crop_pig(img, row["bbox"])
        return self.transform(img), str(row["row_id"])


In [16]:
def run_tta_inference(model, checkpoint_path, label):
    # Checkpoint in das unwrapped Modell laden
    unwrap_model(model).load_state_dict(torch.load(checkpoint_path, map_location=device))
    model.eval()
    logits_dict = {}

    for t_idx, tta_tf in enumerate(tta_transforms):
        print(f"  [{label}] TTA {t_idx+1}/{len(tta_transforms)} ...")
        loader = DataLoader(
            TestDataset(test_df, dir_test, tta_tf),
            batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS
        )
        with torch.no_grad():
            for imgs, rids in tqdm(loader, leave=False):
                imgs = imgs.to(device, non_blocking=True)
                with autocast():
                    logits = model(imgs).float().cpu().numpy()
                for j, rid in enumerate(rids):
                    logits_dict[rid] = logits_dict.get(rid, 0) + logits[j]

    return logits_dict

model_A = wrap_model(PigModel(num_classes=num_classes))
print(f"\n🔍 Modell A Inferenz ({best_A}) ...")
logits_A = run_tta_inference(model_A, best_A, "A")
del model_A
torch.cuda.empty_cache()

  Backbone: convnextv2_huge.fcmae_ft_in22k_in1k_384 | Feature dim: 2816
  ✅ DataParallel aktiviert auf GPUs: [2, 3]

🔍 Modell A Inferenz (cnv2_A_p3.pth) ...
  [A] TTA 1/5 ...


  [A] TTA 2/5 ...


  [A] TTA 3/5 ...


  [A] TTA 4/5 ...


  [A] TTA 5/5 ...


In [17]:
model_B = wrap_model(PigModel(num_classes=num_classes))
print(f"\n🔍 Modell B Inferenz ({best_B}) ...")
logits_B = run_tta_inference(model_B, best_B, "B")

# ── Ensemble ──
print("\n⚙️  Ensemble: Durchschnitt Modell A + Modell B ...")
all_row_ids = sorted(set(logits_A.keys()) | set(logits_B.keys()))

ensemble_preds = {}
for rid in all_row_ids:
    la = logits_A.get(rid, np.zeros(num_classes))
    lb = logits_B.get(rid, np.zeros(num_classes))
    ensemble_preds[rid] = (la / len(tta_transforms) + lb / len(tta_transforms)) / 2.0

submission = pd.DataFrame({
    "row_id":   list(ensemble_preds.keys()),
    "class_id": [int(np.argmax(v)) for v in ensemble_preds.values()]
})
submission.to_csv("ensemble_submission.csv", index=False)
print(f"\n✅ Gespeichert: ensemble_submission.csv ({len(submission)} Zeilen)")
print("\nKlassen-Verteilung Submission:")
print(submission["class_id"].value_counts().sort_index())


  Backbone: convnextv2_huge.fcmae_ft_in22k_in1k_384 | Feature dim: 2816
  ✅ DataParallel aktiviert auf GPUs: [2, 3]

🔍 Modell B Inferenz (cnv2_B_p3.pth) ...
  [B] TTA 1/5 ...


  [B] TTA 2/5 ...


  [B] TTA 3/5 ...


  [B] TTA 4/5 ...


  [B] TTA 5/5 ...



⚙️  Ensemble: Durchschnitt Modell A + Modell B ...

✅ Gespeichert: ensemble_submission.csv (11708 Zeilen)

Klassen-Verteilung Submission:
class_id
0    1007
1     136
2    3566
3    5144
4    1855
Name: count, dtype: int64
